# 🌌 DreamScape AI — Gradio Interface (Deliverable 3)

This notebook loads the core multimodal generation functions from  
`multimodal_generation_d3.ipynb` and exposes them through a Gradio UI.

In [1]:
from pathlib import Path

import gradio as gr

# Ensure we are in the notebooks folder when running this
print("Current working directory:", Path(".").resolve())

# Load all functions: sanitize_prompt, multimodal_with_extras, run_all_with_audio, etc.
%run ./multimodal_generation_d3.ipynb

Current working directory: /Users/saturnine/Dreamscape/notebooks
BASE: /Users/saturnine/Dreamscape/notebooks
OUT_DIR: /Users/saturnine/Dreamscape/notebooks/results


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loaded text→image: stabilityai/sd-turbo on mps (dtype=torch.float32); safety=ON
versions => transformers 4.57.1 | torch 2.9.1 | scipy 1.16.3 | hf-hub 0.36.0


Device set to use cpu
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


🎵 Loaded MusicGen via pipeline: facebook/musicgen-small
🔊 AUDIO_ENGINE active: MusicGen/Transformers(facebook/musicgen-small)


Device set to use cpu


🧩 Motif NER: dslim/bert-base-NER loaded.


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✅ Loaded CLIP model for semantic alignment: openai/clip-vit-base-patch32 on cpu
🎤 Loaded Faster-Whisper: small on cpu (int8)


In [ ]:
with gr.Blocks(title="DreamScape AI — Multimodal Dream Generator") as demo:
    gr.Markdown(
        """
        # 🌌 DreamScape AI — Multimodal Dream Generator (D3)

        Enter or record a dream, and the system will generate:
        - 🎨 A surreal image (Stable Diffusion Turbo)  
        - 🎵 Ambient audio (MusicGen or fallback)  
        - 🧩 A symbol / motif graph (NER + lexicon)  
        - 🎨 A 6-tile moodboard (optional)  
        - 📊 Toxicity score + CLIP text–image alignment score  

        This prototype runs fully on local hardware (CPU / MPS / GPU).
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            tbox = gr.Textbox(
                label="Describe your dream (text input)",
                lines=4,
                placeholder=(
                    "Example: My reflection in the mirror started breathing, "
                    "then turned into a bird flying through a burning city..."
                ),
            )
            aud = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="Optional: record/upload your dream (audio)",
            )
            prefer_audio = gr.Checkbox(
                value=True,
                label="Use audio transcript if audio is provided",
            )
            make_mood = gr.Checkbox(value=True, label="Generate moodboard")
            make_motif = gr.Checkbox(value=True, label="Generate motif graph")
            fast = gr.Checkbox(
                value=False,
                label="Fast mode (smaller image, fewer steps)",
            )
            run = gr.Button("Generate Multimodal Dream", variant="primary")

        with gr.Column(scale=1):
            with gr.Tab("Image & Audio"):
                out_img = gr.Image(label="Generated Image")
                out_aud = gr.Audio(label="Generated Audio")

            with gr.Tab("Moodboard & Motifs"):
                out_mb = gr.Image(label="Moodboard (6 styles)")
                out_g = gr.Image(label="Motif Graph")

            with gr.Tab("Analysis"):
                out_t = gr.Textbox(
                    label="Toxicity, CLIP alignment, runtime",
                    lines=4,
                )
                out_asr = gr.Textbox(
                    label="Transcribed Text (if audio used)",
                    lines=4,
                )

    run.click(
        fn=run_all_with_audio,
        inputs=[tbox, aud, prefer_audio, make_mood, make_motif, fast],
        outputs=[out_img, out_aud, out_mb, out_g, out_t, out_asr],
    )

demo.launch(debug=True)  # set share=True when you want a public 1-week link

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
